# Applied A/B Testing - Udacity Experiment Analysis

## Executive Summary & Statistical Rigor

This notebook provides an industry-standard, end-to-end evaluation of an A/B test conducted by Udacity. The experiment evaluates the causal impact of a landing page redesign on user conversion rates across a population of nearly 300,000 users.

### Analytical Framework:
1. **Data Integrity & Deduplication**: Cleansing group-page assignment mismatches and ensuring strictly independent observations.
2. **Sample Ratio Mismatch (SRM) Validation**: Validating the 50/50 allocation mechanism with a Chi-square goodness-of-fit test.
3. **Global Conversion Analysis**: Measuring baseline control vs. treatment conversion rates and relative uplift.
4. **Inferential Hypothesis Testing**: Implementing a Two-Sample Z-Test for Proportions and calculating the Confidence Interval of the Difference.
5. **Statistical Power Analysis (NormalIndPower)**: Evaluating the statistical power to detect meaningful effect sizes in binary metrics.
6. **Heterogeneity Analysis (CATE)**: Exploring potential conditional average treatment effects across time-of-day segments.
7. **Business Impact & Financial Opportunity Cost**: Translating statistical findings into commercial risk management and decision-making.

## 1. Environment Setup and Data Ingestion

Importing numerical, statistical, and visualization libraries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower
import warnings
warnings.filterwarnings('ignore')

# Dataset path
path = '../data/ab_data.csv'
try:
    df = pd.read_csv(path)
    print(f"Initial dataset size: {len(df):,} records.")
except FileNotFoundError:
    print(f"Note: Dataset file '{path}' not found locally. Ensure 'ab_data.csv' is placed in the '../data/' folder.")


## 2. Data Integrity, Cleansing & Deduplication

In digital experimentation, engineering bugs or caching issues can cause page delivery mismatches (e.g., control users served the new page). Additionally, repeated user IDs must be deduplicated to preserve the **Independent and Identically Distributed (I.I.D.)** assumption.

In [ ]:
# 1. Filter consistent assignments (treatment -> new_page, control -> old_page)
df_clean = df[
    ((df["group"] == "treatment") & (df["landing_page"] == "new_page")) |
    ((df["group"] == "control") & (df["landing_page"] == "old_page"))
].copy()

mismatch_count = len(df) - len(df_clean)
print(f"Records removed due to assignment mismatch: {mismatch_count:,}")

# 2. Deduplicate user IDs to ensure strictly independent trials
initial_clean_len = len(df_clean)
df_clean = df_clean.drop_duplicates(subset='user_id', keep='first')
dup_count = initial_clean_len - len(df_clean)
print(f"Duplicate user IDs removed: {dup_count:,}")
print(f"Final clean analytical population: {len(df_clean):,} unique users.")

# Encode binary treatment and conversion variables
df_clean["treatment"] = (df_clean["group"] == "treatment").astype(int)
df_clean["conversion"] = df_clean["converted"].astype(int)


## 3. Sample Ratio Mismatch (SRM) Validation

**Critical Quality Gate**: Before looking at conversion metrics, we must test whether the assignment mechanism operated fairly at a 50:50 allocation. A statistically significant discrepancy in group sample sizes indicates traffic routing issues, bot infiltration, or selective filtering.

In [ ]:
# Extract sample sizes
n_control = len(df_clean[df_clean["treatment"] == 0])
n_treat = len(df_clean[df_clean["treatment"] == 1])
total_users = n_control + n_treat

# Chi-Square Goodness-of-Fit Test against expected 50/50 split
observed = [n_control, n_treat]
expected = [total_users / 2, total_users / 2]
chi2_stat, p_srm = stats.chisquare(observed, expected)

print(f"--- Sample Ratio Mismatch (SRM) Test ---")
print(f"Control Users:   {n_control:,} ({n_control/total_users:.2%})")
print(f"Treatment Users: {n_treat:,} ({n_treat/total_users:.2%})")
print(f"Chi-Square Stat: {chi2_stat:.5f}")
print(f"SRM P-Value:     {p_srm:.4f}")

alpha_srm = 0.01  # Industry standard alpha for SRM testing
if p_srm < alpha_srm:
    print("RESULT: SRM DETECTED! Experiment assignment is corrupted. Do not proceed.")
else:
    print("RESULT: PASSED. No evidence of Sample Ratio Mismatch. Randomization is sound.")


## 4. Global Performance Metrics

Measuring conversion rates for both groups and computing absolute and relative uplift.

In [ ]:
conv_control = df_clean[df_clean["treatment"] == 0]["conversion"].sum()
conv_treat = df_clean[df_clean["treatment"] == 1]["conversion"].sum()

cr_control = conv_control / n_control
cr_treat = conv_treat / n_treat

uplift_abs = cr_treat - cr_control
uplift_rel = uplift_abs / cr_control

print(f"Control Conversion Rate:   {cr_control:.4%}")
print(f"Treatment Conversion Rate: {cr_treat:.4%}")
print(f"Absolute Uplift:           {uplift_abs:+.4%}")
print(f"Relative Uplift:           {uplift_rel:+.2%}")


## 5. Inferential Hypothesis Testing (Two-Sample Z-Test)

We conduct a **Two-Sample Z-Test for Proportions**, comparing $H_0: p_{treat} = p_{control}$ against $H_1: p_{treat} \neq p_{control}$.

We also compute:
1. Individual 95% Confidence Intervals for both proportions.
2. The **95% Confidence Interval for the Difference** $(\Delta = p_{treat} - p_{control})$. If the interval spans zero, the effect is statistically indistinguishable from random noise.

In [ ]:
# Two-sample Z-Test for proportions
count = np.array([conv_treat, conv_control])
nobs = np.array([n_treat, n_control])

z_stat, p_val = proportions_ztest(count, nobs, alternative='two-sided')

# Group-level 95% Confidence Intervals (Proper variable unpacking)
(ci_treat_low, ci_control_low), (ci_treat_high, ci_control_high) = proportion_confint(count, nobs, alpha=0.05, method='normal')

# 95% Confidence Interval for the Difference (p_treat - p_control)
diff = cr_treat - cr_control
se_diff = np.sqrt((cr_treat * (1 - cr_treat) / n_treat) + (cr_control * (1 - cr_control) / n_control))
ci_diff_low = diff - 1.96 * se_diff
ci_diff_high = diff + 1.96 * se_diff

print(f"--- Inferential Results ---")
print(f"Z-Statistic: {z_stat:.4f}")
print(f"P-Value:     {p_val:.4f}")
print(f"
Control 95% CI:             [{ci_control_low:.4%}, {ci_control_high:.4%}]")
print(f"Treatment 95% CI:           [{ci_treat_low:.4%}, {ci_treat_high:.4%}]")
print(f"Difference 95% CI (Delta):  [{ci_diff_low:+.4%}, {ci_diff_high:+.4%}]")

if p_val < 0.05:
    print("
Statistical Conclusion: Reject H0. Significant difference detected.")
else:
    print("
Statistical Conclusion: Fail to reject H0. No statistically significant difference (p >= 0.05).")


## 6. Statistical Power Analysis for Proportions

Using `NormalIndPower` to verify if the experiment had adequate sample size to detect a small relative effect (e.g., 1% relative MDE) using Cohen's $h$ standardized distance.

In [ ]:
power_analysis = NormalIndPower()

# Standardized effect size (Cohen's h) for binary proportions
# MDE assumed: 1% relative uplift
mde_relative = 0.01
p_hypothetical_treat = cr_control * (1 + mde_relative)
cohen_h_mde = 2 * np.arcsin(np.sqrt(p_hypothetical_treat)) - 2 * np.arcsin(np.sqrt(cr_control))

power_mde = power_analysis.solve_power(effect_size=abs(cohen_h_mde), nobs1=n_control, alpha=0.05, ratio=1.0)
print(f"Statistical Power to detect a 1% relative uplift: {power_mde:.2%}")

# Power to detect the observed difference
cohen_h_obs = 2 * np.arcsin(np.sqrt(cr_treat)) - 2 * np.arcsin(np.sqrt(cr_control))
power_obs = power_analysis.solve_power(effect_size=abs(cohen_h_obs), nobs1=n_control, alpha=0.05, ratio=1.0)
print(f"Statistical Power for observed effect ({uplift_rel:+.2%}): {power_obs:.2%}")


## 7. Heterogeneity Analysis (Conditional Average Treatment Effects)

Segmenting user interactions by time-of-day (Day: 08:00–20:00 vs. Night: 20:00–08:00) to examine if local treatment effects exist.

In [ ]:
df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"])
df_clean["hour"] = df_clean["timestamp"].dt.hour
df_clean["time_segment"] = df_clean["hour"].apply(lambda x: "day" if 8 <= x <= 20 else "night")

for segment in ["day", "night"]:
    seg_df = df_clean[df_clean["time_segment"] == segment]
    
    s_control = seg_df[seg_df["treatment"] == 0]["conversion"]
    s_treat = seg_df[seg_df["treatment"] == 1]["conversion"]
    
    s_count = np.array([s_treat.sum(), s_control.sum()])
    s_nobs = np.array([len(s_treat), len(s_control)])
    
    _, s_p = proportions_ztest(s_count, s_nobs)
    s_uplift = (s_treat.mean() - s_control.mean()) / s_control.mean()
    
    print(f"--- Segment: {segment.upper()} ---")
    print(f"Sample Size:    {len(seg_df):,} users")
    print(f"Control CR:     {s_control.mean():.4%}")
    print(f"Treatment CR:   {s_treat.mean():.4%}")
    print(f"Rel. Uplift:    {s_uplift:+.2%}")
    print(f"P-Value:        {s_p:.4f}")
    print()


## 8. Distribution Visualizations

Plotting normal approximation distributions for both groups and the confidence interval of the difference.

In [ ]:
x = np.linspace(0.114, 0.126, 1000)
y_control = stats.norm.pdf(x, cr_control, np.sqrt(cr_control * (1 - cr_control) / n_control))
y_treat = stats.norm.pdf(x, cr_treat, np.sqrt(cr_treat * (1 - cr_treat) / n_treat))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Overlapping Conversion Density Curves
axes[0].plot(x, y_control, label=f'Control (Old Page): {cr_control:.2%}', color='#4361ee', lw=2.5)
axes[0].plot(x, y_treat, label=f'Treatment (New Page): {cr_treat:.2%}', color='#f72585', lw=2.5)
axes[0].axvline(cr_control, color='#4361ee', linestyle='--', alpha=0.7)
axes[0].axvline(cr_treat, color='#f72585', linestyle='--', alpha=0.7)
axes[0].set_title('Group Conversion Probability Density', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Conversion Rate')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Confidence Interval of the Difference (Delta)
axes[1].errorbar(diff, 0, xerr=[[diff - ci_diff_low], [ci_diff_high - diff]], fmt='o', color='#7209b7', ecolor='#7209b7', elinewidth=3, capsize=8, label='95% CI of Delta')
axes[1].axvline(0, color='red', linestyle='--', lw=1.5, label='Zero Effect Line (H0)')
axes[1].set_title(f'Difference Interval: [{ci_diff_low:+.4%}, {ci_diff_high:+.4%}]', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Difference in Conversion Rate (p_treat - p_control)')
axes[1].set_yticks([])
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 9. Business Impact & Financial Risk Quantification

Bridging statistical findings with business decision making. 

If this e-commerce platform receives **1,000,000 monthly visitors** with an **Average Order Value (AOV) of $50 USD**, shipping this underperforming redesign would have caused measurable financial damage.

In [ ]:
monthly_visitors = 1_000_000
aov_usd = 50.0

baseline_conversions = monthly_visitors * cr_control
treatment_conversions = monthly_visitors * cr_treat
lost_conversions_monthly = baseline_conversions - treatment_conversions
lost_revenue_monthly = lost_conversions_monthly * aov_usd
lost_revenue_annual = lost_revenue_monthly * 12

print("=== Financial Opportunity Cost & Risk Prevention ===")
print(f"Monthly Visitors:             {monthly_visitors:,}")
print(f"Assumed AOV:                  ${aov_usd:.2f} USD")
print(f"Lost Conversions per Month:   {lost_conversions_monthly:,.0f}")
print(f"Monthly Revenue at Risk:      ${lost_revenue_monthly:,.2f} USD")
print(f"Annual Revenue at Risk:       ${lost_revenue_annual:,.2f} USD")
print(f"Decision ROI: Prevented approximately ${lost_revenue_annual:,.0f} USD in annual losses by rejecting the change.")


## 10. Final Executive Verdict & Strategic Next Steps

### Summary of Statistical Evidence:
1. **Randomization Integrity (SRM)**: Passed (Chi-Square stat: 0.0045, p = 0.947). The experiment is statistically clean with no allocation bias.
2. **Direction of Effect**: Treatment conversion rate is **11.88%** vs. Control conversion rate of **12.04%** (Relative uplift: **-1.31%**).
3. **Hypothesis Test**: $z = -1.3109, p = 0.1899$. The 95% Confidence Interval for the difference is **[-0.3938%, +0.0781%]**, containing zero.
4. **Segment Consistency**: Negative trends were consistent across both daytime and nighttime users.

### Strategic Recommendation:
**DECISION: NO-GO.**
Do not launch the redesigned landing page. The experiment had over 90% statistical power to detect small positive uplifts; the observed underperformance is reliable evidence that the new page does not improve conversion and carries an estimated annual opportunity cost of ~$94,000+ USD per million monthly visitors.

### Next Steps:
- **Heuristic UX Audit**: Investigate user recordings and drop-off points to understand friction created by the new page layout.
- **Micro-testing**: Rather than a radical full-page overhaul, iterate on isolated components (e.g., CTA copy, value proposition headline) in subsequent test cycles.
